In [ ]:
# Run this cell first — fixes table and equation alignment throughout the notebook
from IPython.display import display, HTML
display(HTML("<style>table {margin-left: 0 !important;} .MathJax_Display, .MathJax {text-align: left !important;}</style>"))

# Week 6 Lab — Grouped Analysis, Pivot Tables, and the Micro-Project
**Introduction to Python for Business Statistics**

---

### What This Lab Covers
- `groupby` — summarising data by category
- `agg` — applying multiple statistics at once
- Pivot tables — cross-tabulating two categorical variables
- A complete exploratory data analysis (EDA) workflow
- **Micro-project** — end-to-end analysis of a business dataset

**Estimated time:** 90–120 minutes

---

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")

# Build the course dataset — same seed as Week 5 for consistency
np.random.seed(42)
n = 120

df = pd.DataFrame({
    "rep":          np.random.choice(["Chen", "Patel", "Okafor", "Rivera", "Thompson"], n),
    "region":       np.random.choice(["Northeast", "West", "South", "Midwest"], n),
    "quarter":      np.random.choice(["Q1", "Q2", "Q3", "Q4"], n),
    "product":      np.random.choice(["Hardware", "Software", "Services"], n),
    "revenue":      np.round(np.random.normal(42000, 8000, n), 2),
    "units_sold":   np.random.randint(10, 80, n).astype(float),
    "discount_pct": np.round(np.random.uniform(0, 0.30, n), 3)
})

# Clean: fill missing, fix invalid (same steps as Week 5)
df.loc[np.random.choice(df.index, 8, replace=False), "revenue"]      = np.nan
df.loc[np.random.choice(df.index, 5, replace=False), "units_sold"]   = np.nan
df.loc[np.random.choice(df.index, 3, replace=False), "discount_pct"] = -0.05

df["revenue"] = df["revenue"].fillna(df["revenue"].median())
df["units_sold"] = df["units_sold"].fillna(df["units_sold"].median())
df.loc[df["discount_pct"] < 0, "discount_pct"] = 0

df["revenue_per_unit"] = np.where(
    df["units_sold"] > 0,
    df["revenue"] / df["units_sold"],
    np.nan
)

print("Dataset ready. Shape:", df.shape)
df.head()

---
## Part 1 — Grouped Analysis with `groupby`

`groupby` splits a DataFrame by one or more categorical columns and applies an aggregation function to each group.

```python
df.groupby("category")["numeric_column"].aggregation()
```

This answers questions like: *What is the average revenue per region?* or *How many transactions did each rep close?*

In [ ]:
# Mean revenue by region
print(df.groupby("region")["revenue"].mean().round(2))

In [ ]:
# Multiple statistics at once with .agg()
region_summary = df.groupby("region")["revenue"].agg(
    count="count",
    total="sum",
    mean="mean",
    median="median",
    std="std"
).round(2)

print(region_summary)

In [ ]:
# Group by multiple columns
rep_quarter = df.groupby(["rep", "quarter"])["revenue"].mean().round(2)
print(rep_quarter)

### 📊 Stats Connection — Comparing Groups

When comparing means across groups, always look at spread alongside the mean. A region with a higher mean revenue but a much higher standard deviation may be less reliable than one with a slightly lower mean but tight consistency.

The **coefficient of variation (CV)** expresses std as a percentage of the mean — useful for comparing variability across groups with different scales:

$$CV = \frac{s}{\bar{x}} \times 100$$

In [ ]:
# Add coefficient of variation to the region summary
region_cv = df.groupby("region")["revenue"].agg(
    mean="mean",
    std="std"
)
region_cv["cv_pct"] = (region_cv["std"] / region_cv["mean"] * 100).round(1)
print(region_cv)

---
### ✏️ Exercise 1.1 — Grouped Analysis

In [ ]:
# Exercise 1.1

# 1. Calculate total revenue and mean units_sold by product line
product_summary = 
print(product_summary)

# 2. Calculate mean revenue per rep, sorted highest to lowest
rep_mean_revenue = 
print(rep_mean_revenue)

# 3. Calculate the coefficient of variation for revenue by rep
#    Which rep is most consistent?
rep_cv = df.groupby("rep")["revenue"].agg(mean="mean", std="std")
rep_cv["cv_pct"] = 
print(rep_cv.sort_values("cv_pct"))

---
## Part 2 — Pivot Tables

A **pivot table** cross-tabulates two categorical variables and fills each cell with an aggregated value — typically a mean, sum, or count.

```python
df.pivot_table(
    values  = "numeric_column",
    index   = "row_category",
    columns = "column_category",
    aggfunc = "mean"            # or "sum", "count", "median"
)
```

In [ ]:
# Mean revenue by region (rows) and quarter (columns)
pivot_rev = df.pivot_table(
    values="revenue",
    index="region",
    columns="quarter",
    aggfunc="mean"
).round(0)

print(pivot_rev)

In [ ]:
# Visualise the pivot table as a heatmap
fig, ax = plt.subplots(figsize=(8, 4))
sns.heatmap(pivot_rev, annot=True, fmt=".0f", cmap="YlGnBu",
            linewidths=0.5, ax=ax)
ax.set_title("Mean Revenue by Region and Quarter")
plt.tight_layout()
plt.show()

In [ ]:
# Transaction count by rep and product
pivot_count = df.pivot_table(
    values="revenue",
    index="rep",
    columns="product",
    aggfunc="count"
)

print(pivot_count)

---
### ✏️ Exercise 2.1 — Pivot Table Analysis

In [ ]:
# Exercise 2.1

# 1. Create a pivot table showing total revenue by product (rows)
#    and quarter (columns). Use aggfunc="sum".
pivot_product_quarter = 
print(pivot_product_quarter)

# 2. Visualise it as a heatmap with annotations
fig, ax = plt.subplots(figsize=(8, 4))

# your code here

plt.tight_layout()
plt.show()

# 3. Based on the pivot table, which product-quarter combination
#    generated the most revenue? The least?
#

---
## Part 3 — The EDA Workflow

**Exploratory data analysis (EDA)** is a systematic process for understanding a dataset before drawing conclusions. It combines cleaning, statistics, and visualisation into a structured sequence.

A standard EDA workflow has five steps:

| Step | Questions to Answer |
|:-----|:--------------------|
| **1. Understand the structure** | How many rows/columns? What are the data types? |
| **2. Check data quality** | Missing values? Invalid entries? Duplicates? |
| **3. Univariate analysis** | What does each variable look like on its own? |
| **4. Bivariate analysis** | How do variables relate to each other? |
| **5. Grouped analysis** | Do patterns differ across categories? |

The cells below walk through this workflow on the course dataset.

In [ ]:
# Step 1 — Structure
print("Shape:", df.shape)
print("\nData types:")
print(df.dtypes)
print("\nFirst rows:")
df.head()

In [ ]:
# Step 2 — Data quality
print("Missing values:")
print(df.isnull().sum())
print("\nDuplicates:", df.duplicated().sum())
print("\nRevenue range:", df["revenue"].min().round(2), "to", df["revenue"].max().round(2))
print("Invalid discounts:", (df["discount_pct"] < 0).sum())

In [ ]:
# Step 3 — Univariate analysis
print("Summary statistics:")
print(df.describe().round(2))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

sns.histplot(df["revenue"],      bins=20, kde=True, ax=axes[0], color="steelblue")
axes[0].set_title("Revenue Distribution")

sns.histplot(df["units_sold"],   bins=15, kde=True, ax=axes[1], color="coral")
axes[1].set_title("Units Sold Distribution")

sns.histplot(df["discount_pct"], bins=15, kde=True, ax=axes[2], color="mediumseagreen")
axes[2].set_title("Discount % Distribution")

plt.tight_layout()
plt.show()

In [ ]:
# Step 4 — Bivariate analysis
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

sns.scatterplot(data=df, x="units_sold", y="revenue",
                hue="product", alpha=0.7, ax=axes[0])
axes[0].set_title("Units Sold vs Revenue")

numeric_cols = df[["revenue", "units_sold", "discount_pct", "revenue_per_unit"]]
sns.heatmap(numeric_cols.corr().round(2), annot=True, cmap="coolwarm",
            center=0, fmt=".2f", ax=axes[1])
axes[1].set_title("Correlation Heatmap")

plt.tight_layout()
plt.show()

In [ ]:
# Step 5 — Grouped analysis
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

sns.boxplot(data=df, x="region",  y="revenue", palette="Set2", hue="region", legend=False,   ax=axes[0])
axes[0].set_title("Revenue by Region")

sns.boxplot(data=df, x="product", y="revenue", palette="Pastel1", hue="product", legend=False, ax=axes[1])
axes[1].set_title("Revenue by Product")

plt.tight_layout()
plt.show()

---
### ✏️ Exercise 3.1 — EDA Interpretation

Based on the EDA outputs above, answer the following questions in comments.

In [ ]:
# Exercise 3.1 — write your answers as comments

# 1. What is the approximate shape of the revenue distribution?
#    Is the mean close to the median? What does that suggest?
print(f"Revenue mean:   ${df['revenue'].mean():,.0f}")
print(f"Revenue median: ${df['revenue'].median():,.0f}")
#

# 2. Based on the scatter plot, is there a visible linear relationship
#    between units_sold and revenue? What does the correlation value say?
print(f"\nCorrelation (units_sold vs revenue): {df['units_sold'].corr(df['revenue']):.3f}")
#

# 3. From the boxplots, which region shows the widest spread in revenue?
#    Confirm by printing the std for each region.
print("\nRevenue std by region:")
print(df.groupby("region")["revenue"].std().round(2).sort_values(ascending=False))
#

# 4. Which product line has the highest median revenue?
print("\nMedian revenue by product:")
print(df.groupby("product")["revenue"].median().round(2).sort_values(ascending=False))
#

---
## Part 4 — Micro-Project

This is the Week 6 micro-project. Work through it independently — it combines every skill from the course.

---

### The Scenario

You are a junior data analyst at a regional office supply company. The sales director has asked you to analyse the past year of transaction data and produce a summary report answering the following business questions:

1. What does the overall revenue distribution look like?
2. Which region is the strongest performer? Which is most consistent?
3. Which product line drives the most revenue?
4. Is there a relationship between discount level and revenue?
5. Which rep should be highlighted as a top performer?
6. Are there any anomalies in the data worth flagging?

---

### The Dataset

In [ ]:
# Micro-project dataset — a fresh, slightly larger sample
np.random.seed(99)
n = 200

project_df = pd.DataFrame({
    "rep":          np.random.choice(["Chen", "Patel", "Okafor", "Rivera", "Thompson"], n),
    "region":       np.random.choice(["Northeast", "West", "South", "Midwest"], n),
    "quarter":      np.random.choice(["Q1", "Q2", "Q3", "Q4"], n),
    "product":      np.random.choice(["Hardware", "Software", "Services"], n,
                                     p=[0.45, 0.35, 0.20]),
    "revenue":      np.round(np.random.normal(44000, 9500, n), 2),
    "units_sold":   np.random.randint(8, 90, n).astype(float),
    "discount_pct": np.round(np.random.uniform(0, 0.35, n), 3)
})

# Data quality issues to find and fix
project_df.loc[np.random.choice(project_df.index, 10, replace=False), "revenue"]      = np.nan
project_df.loc[np.random.choice(project_df.index, 6,  replace=False), "units_sold"]   = np.nan
project_df.loc[np.random.choice(project_df.index, 4,  replace=False), "discount_pct"] = -0.10
project_df.loc[np.random.choice(project_df.index, 2,  replace=False), "revenue"]      = -5000

print("Project dataset shape:", project_df.shape)
project_df.head()

---
### Step 1 — Understand the Structure

Begin by inspecting the dataset.

In [ ]:
# Step 1 — your code here
# Print shape, dtypes, and the first few rows


### Step 2 — Data Quality and Cleaning

Identify and fix all data quality issues.

In [ ]:
# Step 2 — your code here
# Check for: missing values, negative revenue, invalid discounts, duplicates
# Fix each issue and document what you did as comments


### Step 3 — Univariate Analysis

Examine each variable individually.

In [ ]:
# Step 3 — your code here
# Print summary statistics
# Plot histograms for revenue, units_sold, and discount_pct
# Comment on the shape of each distribution


### Step 4 — Bivariate Analysis

Examine relationships between variables.

In [ ]:
# Step 4 — your code here
# Scatter plot: discount_pct vs revenue
# Correlation matrix heatmap
# Comment: is there a meaningful relationship between discount and revenue?


### Step 5 — Grouped Analysis

Answer the business questions using groupby and pivot tables.

In [ ]:
# Step 5 — your code here

# Q2: Region analysis — mean, median, std, CV for revenue by region

# Q3: Product line analysis — total and mean revenue by product

# Q5: Rep analysis — mean revenue and transaction count by rep

# Pivot table: mean revenue by region and product

# Boxplots: revenue by region AND revenue by product (side by side)


### Step 6 — Outlier and Anomaly Check

Flag any unusual values.

In [ ]:
# Step 6 — your code here
# Use the IQR method to detect outliers in revenue
# Use z-scores as a second check
# Report what you found


### Step 7 — Summary Report

Write up your findings. Answer each of the six business questions in plain language.

In [ ]:
print("====== SALES ANALYSIS REPORT ======")
print()

# Q1: Revenue distribution
print("1. Revenue Distribution")
# your summary here
print()

# Q2: Strongest and most consistent region
print("2. Regional Performance")
# your summary here
print()

# Q3: Revenue by product line
print("3. Product Line Performance")
# your summary here
print()

# Q4: Discount vs revenue relationship
print("4. Discount Analysis")
# your summary here
print()

# Q5: Top performing rep
print("5. Rep Performance")
# your summary here
print()

# Q6: Anomalies
print("6. Data Quality and Anomalies")
# your summary here

---
## Week 6 Summary

| Concept | Key Point |
|:--------|:----------|
| `groupby` | Split-apply-combine — group by category, apply a function |
| `.agg()` | Apply multiple aggregations in one call |
| Coefficient of variation | $s / \bar{x} \times 100$ — relative spread, comparable across groups |
| Pivot table | Cross-tabulate two categoricals — rows, columns, aggregated values |
| EDA workflow | Structure → quality → univariate → bivariate → grouped |
| Analysis narrative | Numbers alone are not findings — interpretation is the job |

---

### Course Complete

Over six weeks you have built a working foundation in both Python and business statistics:

| Week | Python | Statistics |
|:-----|:-------|:-----------|
| 1 | Variables, types, arithmetic | Data types, measurement scales |
| 2 | Lists, NumPy arrays | Central tendency, samples vs populations |
| 3 | Conditionals, loops, comprehensions | Frequency distributions, outlier detection |
| 4 | Functions, error handling, Pandas | Variance, std deviation, z-scores, IQR |
| 5 | Data cleaning, Matplotlib, Seaborn | Distributions, correlation, visualisation |
| 6 | groupby, pivot tables, EDA workflow | Grouped analysis, coefficient of variation |

The micro-project is the first time you have applied all of these together on a single dataset — exactly how real analysis works.